<a href="https://colab.research.google.com/github/Guru-4747/Heart-Health-RAG-Chatbot/blob/main/Heart_Health_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install dependencies

In [ ]:
!pip install -q requests beautifulsoup4 sentence-transformers transformers accelerate faiss-cpu ipywidgets


imports and configuration

In [ ]:
import re
import html
import gc
from dataclasses import dataclass
from typing import List, Dict, Tuple

import requests
import numpy as np
import faiss
import torch
import ipywidgets as widgets

from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, Markdown

# ---------------------------
# Models
# ---------------------------
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATION_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# ---------------------------
# RAG settings
# ---------------------------
CHUNK_SIZE = 850
CHUNK_OVERLAP = 140
TOP_K = 4
MAX_HISTORY_TURNS = 4
MAX_NEW_TOKENS = 260

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

print(f"Runtime device: {DEVICE}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"LLM: {GENERATION_MODEL}")


Runtime device: cuda
Embedding model: sentence-transformers/all-MiniLM-L6-v2
LLM: Qwen/Qwen2.5-1.5B-Instruct


PUBLIC DATA RESOURCES

In [ ]:
DATA_SOURCES = {
    "NHLBI — Heart-Healthy Living":
        "https://www.nhlbi.nih.gov/health/heart-healthy-living",

    "NHLBI — Heart Disease Risk Factors":
        "https://www.nhlbi.nih.gov/health/heart-healthy-living/risks",

    "NHLBI — Heart-Healthy Foods":
        "https://www.nhlbi.nih.gov/health/heart-healthy-living/healthy-foods",

    "NHLBI — Physical Activity":
        "https://www.nhlbi.nih.gov/health/heart-healthy-living/physical-activity",

    "NHLBI — Manage Stress":
        "https://www.nhlbi.nih.gov/health/heart-healthy-living/manage-stress",

    "NHLBI — Healthy Sleep":
        "https://www.nhlbi.nih.gov/health/heart-healthy-living/sleep",
}

import requests
from bs4 import BeautifulSoup
import html
import re

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120 Safari/537.36"
    )
}


def fetch_clean_text(url: str) -> str:
    """
    Download a public webpage and extract readable text.
    """

    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Remove webpage elements that are not useful for RAG
    for tag in soup([
        "script",
        "style",
        "nav",
        "footer",
        "header",
        "form",
        "noscript",
        "svg"
    ]):
        tag.decompose()

    # Prefer the page's primary content
    main_content = (
        soup.find("main")
        or soup.find("article")
        or soup.body
        or soup
    )

    text = main_content.get_text(
        separator=" ",
        strip=True
    )

    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


documents = []

for title, url in DATA_SOURCES.items():

    try:
        text = fetch_clean_text(url)

        if len(text) < 300:
            print(f"Skipped: {title} — insufficient text")
            continue

        documents.append({
            "title": title,
            "url": url,
            "text": text
        })

        print(
            f"✓ Loaded: {title} "
            f"({len(text):,} characters)"
        )

    except requests.exceptions.HTTPError as error:

        print(
            f"✗ Skipped: {title} — "
            f"HTTP error: {error}"
        )

    except Exception as error:

        print(
            f"✗ Skipped: {title} — {error}"
        )


if not documents:
    raise RuntimeError(
        "No public sources could be loaded."
    )


print(
    f"\n✓ Knowledge base pages loaded: "
    f"{len(documents)}"
)

✓ Loaded: NHLBI — Heart-Healthy Living (1,356 characters)
✓ Loaded: NHLBI — Heart Disease Risk Factors (5,836 characters)
✓ Loaded: NHLBI — Heart-Healthy Foods (7,707 characters)
✓ Loaded: NHLBI — Physical Activity (2,672 characters)
✓ Loaded: NHLBI — Manage Stress (1,491 characters)
✓ Loaded: NHLBI — Healthy Sleep (2,571 characters)

✓ Knowledge base pages loaded: 6


CHUNCK THE DOCUMENTS

In [ ]:
@dataclass
class Chunk:
    title: str
    url: str
    chunk_id: int
    text: str


def split_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    """Split text into overlapping passages while preferring sentence boundaries."""
    pieces = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))

        if end < len(text):
            sentence_end = text.rfind(". ", start, end)
            word_end = text.rfind(" ", start, end)
            boundary = max(sentence_end, word_end)
            if boundary > start + chunk_size // 2:
                end = boundary + 1

        piece = text[start:end].strip()
        if piece:
            pieces.append(piece)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)

    return pieces


chunks: List[Chunk] = []
for doc in documents:
    for chunk_id, text_piece in enumerate(split_text(doc["text"]), start=1):
        chunks.append(
            Chunk(
                title=doc["title"],
                url=doc["url"],
                chunk_id=chunk_id,
                text=text_piece,
            )
        )

print(f"Created {len(chunks)} searchable passages from {len(documents)} pages.")


Created 32 searchable passages from 6 pages.


BUILD THE VECTOR INDEX

In [ ]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

chunk_texts = [chunk.text for chunk in chunks]
chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

vector_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
vector_index.add(chunk_embeddings)

print(f"FAISS index ready: {vector_index.ntotal} vectors × {chunk_embeddings.shape[1]} dimensions")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

FAISS index ready: 32 vectors × 384 dimensions


RETRIVAL FUNCTION

In [ ]:
def retrieve(question: str, top_k: int = TOP_K) -> List[Dict]:
    question = question.strip()
    if not question:
        return []

    query_vector = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype("float32")

    k = min(top_k, len(chunks))
    scores, indices = vector_index.search(query_vector, k)

    results = []
    for score, index in zip(scores[0], indices[0]):
        if index < 0:
            continue
        chunk = chunks[int(index)]
        results.append({
            "title": chunk.title,
            "url": chunk.url,
            "chunk_id": chunk.chunk_id,
            "text": chunk.text,
            "score": float(score),
        })

    return results


def show_retrieval(question: str, top_k: int = TOP_K):
    """Optional debugging helper: inspect what RAG retrieved before generation."""
    for i, item in enumerate(retrieve(question, top_k), start=1):
        print(f"[{i}] {item['title']} | similarity={item['score']:.3f}")
        print(item["text"][:350], "\n")


show_retrieval("What are common risk factors for heart disease?", top_k=3)


[1] NHLBI — Heart Disease Risk Factors | similarity=0.676
alth is understanding your risk of heart disease . Your risk depends on many factors, some of which are changeable and others that are not. Risk factors are conditions or habits that make a person more likely to develop a disease. These risk factors may be different for each person. Preventing heart disease starts with knowing what your risks facto 

[2] NHLBI — Heart-Healthy Living | similarity=0.534
our heart and stay healthy. Heart-healthy living involves understanding your risk , making healthy choices, and taking steps to reduce your chances of getting heart disease, including coronary heart disease , the most common type. By taking preventive measures, you can lower your risk of developing heart disease that could lead to a heart attack. Y 

[3] NHLBI — Heart Disease Risk Factors | similarity=0.516
ster was diagnosed before age 65 Have a history of preeclampsia, which is a sudden rise in blood pressure and too much protein

LOAD THE INSTRUCTION TUNED LLM

In [ ]:
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

llm_tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)

model_kwargs = {
    "torch_dtype": DTYPE,
    "low_cpu_mem_usage": True,
}

if DEVICE == "cuda":
    model_kwargs["device_map"] = "auto"

llm = AutoModelForCausalLM.from_pretrained(
    GENERATION_MODEL,
    **model_kwargs,
)

if DEVICE == "cpu":
    llm = llm.to("cpu")

llm.eval()
print("LLM loaded successfully.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

LLM loaded successfully.


CONVERSATIONAL RAG PIPELINE

In [ ]:
SYSTEM_PROMPT = """You are a heart-health educational assistant using Retrieval-Augmented Generation (RAG).

Rules:
- Base factual medical information on the RETRIEVED CONTEXT supplied for this turn.
- If the retrieved context does not support an answer, clearly say that the available sources do not contain enough information.
- Do not invent diagnoses, test results, medications, dosages, or treatment plans.
- For emergencies or severe symptoms, advise the user to seek urgent professional medical care.
- Keep answers clear and conversational.
- When useful, refer to retrieved passages as [Source 1], [Source 2], and so on.
"""


def build_context(retrieved: List[Dict]) -> str:
    blocks = []
    for i, item in enumerate(retrieved, start=1):
        blocks.append(
            f"[Source {i}] {item['title']}\n"
            f"URL: {item['url']}\n"
            f"Passage: {item['text']}"
        )
    return "\n\n".join(blocks)


def history_to_text(history: List[Dict], max_turns: int = MAX_HISTORY_TURNS) -> str:
    if not history:
        return "No previous conversation."

    recent = history[-(max_turns * 2):]
    lines = []
    for message in recent:
        role = "User" if message["role"] == "user" else "Assistant"
        lines.append(f"{role}: {message['content']}")
    return "\n".join(lines)


def generate_with_llm(messages: List[Dict]) -> str:
    """Generate one answer from the local instruction-tuned LLM."""
    prompt = llm_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = llm_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=min(getattr(llm_tokenizer, "model_max_length", 8192), 8192),
    )

    target_device = next(llm.parameters()).device
    model_inputs = {key: value.to(target_device) for key, value in model_inputs.items()}

    with torch.inference_mode():
        output_ids = llm.generate(
            **model_inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    generated = output_ids[0][model_inputs["input_ids"].shape[1]:]
    answer = llm_tokenizer.decode(generated, skip_special_tokens=True).strip()
    return answer


def rag_answer(question: str, history: List[Dict] = None, top_k: int = TOP_K) -> Tuple[str, List[Dict]]:
    """Complete conversational RAG turn: retrieve -> augment -> LLM generate."""
    question = question.strip()
    history = history or []

    if not question:
        return "Please enter a question.", []

    retrieved = retrieve(question, top_k=top_k)
    context = build_context(retrieved)
    recent_history = history_to_text(history)

    user_prompt = f"""RECENT CONVERSATION:
{recent_history}

RETRIEVED CONTEXT:
{context}

CURRENT USER QUESTION:
{question}

Answer the current question using the retrieved context. If the question refers to something from the recent conversation, use that history only to understand the reference; keep medical facts grounded in the retrieved context.
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    answer = generate_with_llm(messages)

    # Deduplicate page-level citations while preserving retrieval order.
    seen = set()
    sources = []
    for item in retrieved:
        if item["url"] not in seen:
            seen.add(item["url"])
            sources.append({
                "title": item["title"],
                "url": item["url"],
                "score": item["score"],
            })

    return answer, sources


ONE TURN RAG TEST

In [ ]:
test_answer, test_sources = rag_answer("What lifestyle habits can help reduce heart disease risk?")
print(test_answer)
print("\nSources:")
for source in test_sources:
    print("-", source["title"], "—", source["url"])


To reduce heart disease risk, it's important to understand your personal risk factors and make healthy choices. According to the retrieved context, several lifestyle habits can help lower your risk:

1. **Understand Your Risk**: Knowing your risk factors is crucial. Risk factors include high blood pressure, high blood cholesterol, overweight or obesity, prediabetes or diabetes, smoking, lack of regular physical activity, having a family history of early heart disease, and a history of preeclampsia.

2. **Make Healthy Choices**: Making healthy choices can significantly reduce your risk of heart disease. This includes:
   - **Eating a Heart-Healthy Diet**: Consuming foods rich in nutrients such as fruits, vegetables, whole grains, lean proteins, and healthy fats.
   - **Maintaining a Healthy Weight**: Being overweight or obese increases your risk of heart disease.
   - **Regular Physical Activity**: Engaging in regular physical activity helps maintain a healthy weight and reduces the ris

INTERACTIVE NOTE BOOK CHAT

In [ ]:
chat_history: List[Dict] = []

question_box = widgets.Textarea(
    placeholder="Ask a heart-health question...",
    layout=widgets.Layout(width="95%", height="80px"),
)

ask_button = widgets.Button(
    description="Ask RAG",
    button_style="primary",
    tooltip="Retrieve evidence and generate an LLM answer",
)

clear_button = widgets.Button(
    description="New conversation",
    tooltip="Clear the conversation history",
)

status = widgets.HTML(value="<small>Ready — answers use retrieval + the local LLM.</small>")
chat_output = widgets.Output(
    layout=widgets.Layout(
        border="1px solid #888",
        padding="12px",
        width="95%",
        min_height="180px",
    )
)


def render_turn(question: str, answer: str, sources: List[Dict]):
    with chat_output:
        display(Markdown(f"### You\n{question}"))
        display(Markdown(f"### Assistant\n{answer}"))

        if sources:
            source_lines = ["### Retrieved sources"]
            for i, source in enumerate(sources, start=1):
                source_lines.append(
                    f"{i}. [{source['title']}]({source['url']}) "
                    f"— similarity `{source['score']:.3f}`"
                )
            display(Markdown("\n".join(source_lines)))

        display(Markdown("---"))


def handle_question(_=None):
    question = question_box.value.strip()
    if not question:
        status.value = "<small>Please enter a question first.</small>"
        return

    ask_button.disabled = True
    clear_button.disabled = True
    status.value = "<small>Retrieving passages and generating with the LLM...</small>"

    try:
        # IMPORTANT: retrieval + LLM generation happens on every turn.
        answer, sources = rag_answer(question, history=chat_history)

        chat_history.append({"role": "user", "content": question})
        chat_history.append({"role": "assistant", "content": answer})

        render_turn(question, answer, sources)
        question_box.value = ""
        status.value = f"<small>Ready — conversation turns stored: {len(chat_history) // 2}</small>"
    except Exception as exc:
        with chat_output:
            display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))
        status.value = "<small>The last turn failed. Check the error above.</small>"
    finally:
        ask_button.disabled = False
        clear_button.disabled = False


def clear_conversation(_=None):
    chat_history.clear()
    chat_output.clear_output()
    question_box.value = ""
    status.value = "<small>New conversation started. RAG knowledge base is still loaded.</small>"


ask_button.on_click(handle_question)
clear_button.on_click(clear_conversation)

display(
    widgets.VBox([
        widgets.HTML("<h3>Heart Health — Conversational LLM + RAG</h3>"),
        widgets.HTML("<p>Runs in this notebook. No Streamlit, Gradio, Flask, or deployment server.</p>"),
        question_box,
        widgets.HBox([ask_button, clear_button]),
        status,
        chat_output,
    ])
)
